# 🎓 Student Performance Prediction — **Web UI (Frontend)**

Interactive form-based interface built with **Gradio** — enter a student's
previous data and get an instant **Pass/Fail** prediction + **expected score (0–20)**.

| | |
|---|---|
| **Frontend** | Gradio (runs inside Colab — no install needed) |
| **Backend** | Python + scikit-learn (same models as the main notebook) |
| **Models** | Logistic Regression (classifier, 92.8%) + Random Forest (regressor, R²=0.896) |
| **Dataset** | UCI/Kaggle Student Performance — 1044 students |

**How to use:** `Runtime → Run all` → wait for the link → **click the link** →
the form opens → fill details (or drag the G1/G2 sliders) → **PREDICT**.

In [1]:
# ============================================================
# Cell 1 — Make sure Gradio is installed
# ============================================================
import sys, subprocess
try:
    import gradio as gr
except ImportError:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "gradio"])
    import gradio as gr
print("Gradio version:", gr.__version__)

C:\Users\dell\AppData\Local\Programs\Python\Python313\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Gradio version: 6.32.0


In [2]:
# ============================================================
# Cell 2 — Dataset + Train the two best models
#           (same logic & random_state=42 as the main notebook)
# ============================================================
import os, urllib.request, zipfile, warnings
warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split, cross_val_predict
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import accuracy_score, r2_score, mean_squared_error

RANDOM_STATE = 42
DATA_DIR = "data"
MAT_PATH = os.path.join(DATA_DIR, "student-mat.csv")
POR_PATH = os.path.join(DATA_DIR, "student-por.csv")

# --- download dataset if not present (auto in Colab) ---
if not (os.path.exists(MAT_PATH) and os.path.exists(POR_PATH)):
    os.makedirs(DATA_DIR, exist_ok=True)
    url = "https://archive.ics.uci.edu/static/public/320/student+performance.zip"
    zp = os.path.join(DATA_DIR, "sp.zip")
    urllib.request.urlretrieve(url, zp)
    with zipfile.ZipFile(zp) as z:
        z.extractall(DATA_DIR)
    nested = os.path.join(DATA_DIR, "student.zip")
    if os.path.exists(nested):
        with zipfile.ZipFile(nested) as z:
            z.extractall(DATA_DIR)

mat = pd.read_csv(MAT_PATH, sep=";")
por = pd.read_csv(POR_PATH, sep=";")
mat["subject"] = "math"; por["subject"] = "portuguese"
df = pd.concat([mat, por], ignore_index=True)
df["result"] = np.where(df["G3"] >= 10, "Pass", "Fail")
print("Dataset:", df.shape)

# --- features / split (same as notebook) ---
X_raw = df.drop(columns=["G3", "result"])
cat_cols = X_raw.select_dtypes(include="object").columns.tolist()
X = pd.get_dummies(X_raw, columns=cat_cols)
X_train, X_test, yc_train, yc_test, yr_train, yr_test = train_test_split(
    X, df["result"], df["G3"], test_size=0.2,
    random_state=RANDOM_STATE, stratify=df["result"])

# --- best classifier: Logistic Regression + CV threshold ---
clf = Pipeline([("scaler", StandardScaler()),
                ("clf", LogisticRegression(max_iter=1000))])
clf.fit(X_train, yc_train)
oof = cross_val_predict(clf, X_train, yc_train, cv=5, method="predict_proba")[:, 1]
best_t, best_acc = 0.5, 0.0
for t in np.arange(0.30, 0.70, 0.01):
    acc = accuracy_score(yc_train, np.where(oof >= t, "Pass", "Fail"))
    if acc > best_acc:
        best_t, best_acc = t, acc
clf_acc = accuracy_score(yc_test,
    np.where(clf.predict_proba(X_test)[:, 1] >= best_t, "Pass", "Fail"))

# --- best regressor: Random Forest ---
reg = RandomForestRegressor(n_estimators=100, random_state=RANDOM_STATE)
reg.fit(X_train, yr_train)
reg_r2 = r2_score(yr_test, reg.predict(X_test))
reg_rmse = float(np.sqrt(mean_squared_error(yr_test, reg.predict(X_test))))

COLUMNS = list(X.columns)
print(f"Classifier accuracy: {clf_acc*100:.2f}% | threshold: {best_t:.2f}")
print(f"Regressor R2: {reg_r2:.3f} | RMSE: {reg_rmse:.2f} marks")

Dataset: (1044, 35)


Classifier accuracy: 92.82% | threshold: 0.33
Regressor R2: 0.896 | RMSE: 1.24 marks


In [3]:
# ============================================================
# Cell 3 — Prediction function (the logic behind the button)
# ============================================================
def predict(student: dict):
    row = pd.DataFrame([student])
    enc = pd.get_dummies(row).reindex(columns=COLUMNS, fill_value=0)
    proba_pass = float(clf.predict_proba(enc)[0][list(clf.classes_).index("Pass")])
    result = "pass" if proba_pass >= best_t else "fail"
    score = float(np.clip(reg.predict(enc)[0], 0, 20))
    if score >= 16:
        advice = "Excellent! Keep this performance up."
    elif score >= 10:
        advice = "Likely to pass. Improve G1/G2 to score above 15."
    else:
        advice = "High risk of failing. More study time, fewer absences."
    return result, proba_pass, round(score, 2), advice

def predict_ui(school, sex, age, address, famsize, Pstatus, Medu, Fedu,
               Mjob, Fjob, reason, guardian, studytime, failures, schoolsup,
               famsup, paid, activities, nursery, higher, internet, romantic,
               famrel, freetime, goout, Dalc, Walc, health, absences,
               G1, G2, subject):
    """Maps every form field -> one student dict -> model prediction."""
    student = dict(school=school, sex=sex, age=int(age), address=address,
                   famsize=famsize, Pstatus=Pstatus, Medu=int(Medu), Fedu=int(Fedu),
                   Mjob=Mjob, Fjob=Fjob, reason=reason, guardian=guardian,
                   studytime=int(studytime), failures=int(failures),
                   schoolsup=schoolsup, famsup=famsup, paid=paid,
                   activities=activities, nursery=nursery, higher=higher,
                   internet=internet, romantic=romantic, famrel=int(famrel),
                   freetime=int(freetime), goout=int(goout), Dalc=int(Dalc),
                   Walc=int(Walc), health=int(health), absences=int(absences),
                   G1=int(G1), G2=int(G2), subject=subject)
    result, proba, score, advice = predict(student)
    label = {"PASS": proba} if result == "pass" else {"FAIL": 1 - proba}
    return label, score, advice
print("Prediction function ready.")

Prediction function ready.


In [ ]:
# ============================================================
# Cell 4 — Build the Gradio web interface (frontend)
# ============================================================
acc_txt = f"**Model:** Logistic Regression **{clf_acc*100:.1f}%** accuracy · Random Forest **R²={reg_r2:.3f}** · {len(df)} students · {X.shape[1]} features"

# Family and social attributes are no longer asked in the form; they are fixed
# to the demo student's values, so predictions are unchanged.
FIXED_FAMILY = [4, 3, "teacher", "services", "course", "mother"]
FIXED_SOCIAL = ["no", 4, 3, 2, 1, 1, 4]   # romantic, famrel, freetime, goout, Dalc, Walc, health

with gr.Blocks(title="Student Performance Prediction") as demo:
    gr.Markdown("# 🎓 Student Performance Prediction")
    gr.Markdown("Enter the student's **previous data** (personal details, study habits and past marks) "
                "→ get **Pass/Fail** + **expected score** instantly.")
    gr.Markdown(acc_txt)
    gr.Markdown("ℹ️ *Family and social attributes are kept fixed at typical values and are not asked here.*")

    with gr.Accordion("👤 Personal", open=False):
        with gr.Row():
            school   = gr.Dropdown(["GP", "MS"], value="GP", label="School")
            sex      = gr.Dropdown(["M", "F"], value="M", label="Gender")
            age      = gr.Slider(15, 22, value=16, step=1, label="Age")
            address  = gr.Dropdown(["U", "R"], value="U", label="Address (U=urban/R=rural)")
            famsize  = gr.Dropdown(["GT3", "LE3"], value="GT3", label="Family size")
            Pstatus  = gr.Dropdown(["T", "A"], value="T", label="Parent status (T=together/A=apart)")

    with gr.Accordion("📚 Study", open=False):
        with gr.Row():
            studytime = gr.Dropdown([1, 2, 3, 4], value=3, label="Study time (1=<2hrs ... 4=>10hrs)")
            failures  = gr.Dropdown([0, 1, 2, 3], value=0, label="Past class failures")
            absences  = gr.Slider(0, 60, value=2, step=1, label="School absences")
            schoolsup = gr.Dropdown(["no", "yes"], value="no", label="Extra school support")
        with gr.Row():
            famsup     = gr.Dropdown(["yes", "no"], value="yes", label="Family support")
            paid       = gr.Dropdown(["yes", "no"], value="yes", label="Extra paid classes")
            activities = gr.Dropdown(["yes", "no"], value="yes", label="Extra-curriculars")
            nursery    = gr.Dropdown(["yes", "no"], value="yes", label="Attended nursery")
        with gr.Row():
            higher  = gr.Dropdown(["yes", "no"], value="yes", label="Wants higher education")
            internet = gr.Dropdown(["yes", "no"], value="yes", label="Internet at home")

    with gr.Accordion("📝 Previous Marks  ⭐ KEY FEATURES", open=True):
        gr.Markdown("These two previous exam scores matter the most (0–20 each):")
        with gr.Row():
            G1      = gr.Slider(0, 20, value=15, step=1, label="1st exam marks (G1)")
            G2      = gr.Slider(0, 20, value=16, step=1, label="2nd exam marks (G2)")
            subject = gr.Dropdown(["math", "portuguese"], value="math", label="Subject")

    btn = gr.Button("🔮  PREDICT RESULT", variant="primary", size="lg")
    with gr.Row():
        label_out = gr.Label(label="Pass / Fail prediction")
        score_out = gr.Number(label="Predicted score (0-20)")
    advice_out = gr.Textbox(label="Advice", interactive=False)

    # predict_ui still takes all 32 attributes; the hidden ones come in as constants
    inputs = [school, sex, age, address, famsize, Pstatus,
              *FIXED_FAMILY,
              studytime, failures, schoolsup, famsup, paid, activities, nursery, higher, internet,
              *FIXED_SOCIAL,
              absences, G1, G2, subject]
    btn.click(fn=predict_ui, inputs=inputs,
              outputs=[label_out, score_out, advice_out])

print("Interface built.")


In [5]:
# ============================================================
# Cell 5 — Test the exact function wired to the button
# ============================================================
strong = predict_ui("GP", "M", 16, "U", "GT3", "T", 4, 3, "teacher", "services",
                    "course", "mother", 3, 0, "no", "yes", "yes", "yes", "yes",
                    "yes", "yes", "no", 4, 3, 2, 1, 1, 4, 2, 15, 16, "math")
weak = predict_ui("MS", "F", 18, "R", "LE3", "A", 1, 1, "at_home", "other",
                  "reputation", "other", 1, 2, "yes", "no", "no", "no", "no",
                  "yes", "no", "yes", 2, 4, 5, 3, 4, 3, 12, 8, 7, "math")
print("Strong student ->", strong[0], "| score:", strong[1], "|", strong[2])
print("Weak student   ->", weak[0], "| score:", weak[1], "|", weak[2])
assert "PASS" in strong[0] and abs(strong[1] - 16.37) < 0.5, "strong student test failed"
assert "FAIL" in weak[0] and abs(weak[1] - 7.38) < 0.5, "weak student test failed"
print("UI FUNCTION TEST PASSED")

Strong student -> {'PASS': 0.9999902207133303} | score: 16.37 | Excellent! Keep this performance up.
Weak student   -> {'FAIL': 0.9810790859063819} | score: 7.38 | High risk of failing. More study time, fewer absences.
UI FUNCTION TEST PASSED


In [6]:
# ============================================================
# Cell 6 — Launch the web UI
#   -> In Colab a public SHARE LINK is printed; click it to open the form.
#   -> Locally it runs at http://localhost:7860
# ============================================================
import sys
IN_COLAB = "google.colab" in sys.modules
demo.launch(share=IN_COLAB, prevent_thread_lock=True)
if not IN_COLAB:
    print("Open the UI at: http://localhost:7860")

* Running on local URL:  http://127.0.0.1:7860


* To create a public link, set `share=True` in `launch()`.


Open the UI at: http://localhost:7860
